# Hippo Encoder Text Distillation

Google Colab notebook for training the text-only distillation setup in this repo.

## Flow

1. Mount Google Drive
2. Clone or reuse the repo
3. Install dependencies
4. Download a public text corpus
5. Point the config at the generated JSONL
6. Start training

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/CameronBadman/Hippo-encoder.git'
WORKDIR = Path('/content/Hippo-encoder')

if WORKDIR.exists():
    !rm -rf /content/Hippo-encoder

!git clone {REPO_URL} {WORKDIR}
%cd /content/Hippo-encoder


In [ ]:
%cd /content/Hippo-encoder
!python -m pip install --upgrade pip
!pip install -e .


In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/hippo_encoder_data')
TRAIN_JSONL = DATA_ROOT / 'train.jsonl'
OUTPUT_DIR = Path('/content/drive/MyDrive/hippo_encoder_runs/distill-text-tiny')

DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Dataset root:', DATA_ROOT)
print('Train manifest:', TRAIN_JSONL)
print('Output dir:', OUTPUT_DIR)


In [ ]:
%cd /content/Hippo-encoder
!python scripts/prepare_text_dataset.py \
  --dataset ag_news \
  --split train \
  --text-column text \
  --limit 5000 \
  --prefix 'passage: ' \
  --output /content/drive/MyDrive/hippo_encoder_data/train.jsonl


In [ ]:
from pathlib import Path
import json

config = {
    'teacher_model_name': 'intfloat/e5-base-v2',
    'student_model_name': 'distilgpt2',
    'dataset_jsonl': '/content/drive/MyDrive/hippo_encoder_data/train.jsonl',
    'output_dir': '/content/drive/MyDrive/hippo_encoder_runs/distill-text-tiny',
    'max_text_length': 64,
    'batch_size': 8,
    'num_epochs': 1,
    'learning_rate': 1e-4,
    'weight_decay': 1e-2,
    'log_every': 1,
    'save_every': 500,
    'num_workers': 2,
    'teacher_text_weight': 1.0,
    'hidden_state_weight': 0.2,
    'contrastive_weight': 0.2,
    'normalize_targets': True,
    'gradient_clip_norm': 1.0,
    'warmup_steps': 10,
    'seed': 42
}

runtime_config_path = Path('/content/Hippo-encoder/configs/colab_run.json')
runtime_config_path.parent.mkdir(parents=True, exist_ok=True)
runtime_config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
print(runtime_config_path.read_text())


In [ ]:
from pathlib import Path
import json

manifest = Path('/content/drive/MyDrive/hippo_encoder_data/train.jsonl')
rows = [json.loads(line) for line in manifest.read_text().splitlines() if line.strip()]
print('rows:', len(rows))
print('first row:', rows[0])
print('last row:', rows[-1])


In [ ]:
%cd /content/Hippo-encoder
!python -m hippo_encoder.train --config configs/colab_run.json


## Notes

- E5 works best when your texts use task prefixes such as `query:` or `passage:`.
- The notebook currently bootstraps from `ag_news` to get a few thousand statements quickly.
- You can swap in a different public dataset by changing the prep script arguments.
- If you switch to a different student under 1B params, only the config needs to change.